In [14]:
import mne
import os.path
from autoreject import AutoReject
mne.set_log_level('error')
import matplotlib.pyplot as plt

In [15]:
labels = {'brake', 'throttle', 'change', 'turn'}

In [16]:
# for i in labels:
#     for j in range(1, 31):
#         print(f'reading {j}_{i}.set')
#         path = f'/run/media/aaditya/DATA/Datasets/22193119/Rawdata/EEG/{j}_{i}.set'
#         if i in raws.keys():
#             raws[i].append(mne.io.read_raw_eeglab(path))
#         else:
#             raws[i] = [mne.io.read_raw_eeglab(path)]

In [17]:
print('initial preprocessing\n')

for i in range(1, 31):
    for j in labels:
        root = '/run/media/aaditya/DATA/Datasets/'
        file = f'{i}_{j}'

        print(f'reading {file}')

        filepath = root + 'rawdata/Rawdata/EEG/' + file + '.set'
        outpath = root + 'intermediate/pre_epoch/' + file + '_eeg.fif'

        if os.path.isfile(outpath):
            print(f'{file}_eeg.fif already exists, skipping')
            continue

        raw = mne.io.read_raw_eeglab(filepath)

        #drop non EEG-channels
        raw.drop_channels(['ECG', 'HEOR', 'HEOL', 'VEOU', 'VEOL'])
        #set montage to standard 10-05 layout
        raw.set_montage('standard_1005')
        #apply average rereferencing
        raw.set_eeg_reference()
        #apply bandpass filter from 0.5Hz to 40Hz
        raw.filter(0.5, 40)

        raw.save(outpath, overwrite=True)
        print('done')

initial preprocessing

reading 1_change
1_change_eeg.fif already exists, skipping
reading 1_throttle
1_throttle_eeg.fif already exists, skipping
reading 1_brake
1_brake_eeg.fif already exists, skipping
reading 1_turn
1_turn_eeg.fif already exists, skipping
reading 2_change
2_change_eeg.fif already exists, skipping
reading 2_throttle
2_throttle_eeg.fif already exists, skipping
reading 2_brake
2_brake_eeg.fif already exists, skipping
reading 2_turn
2_turn_eeg.fif already exists, skipping
reading 3_change
3_change_eeg.fif already exists, skipping
reading 3_throttle
3_throttle_eeg.fif already exists, skipping
reading 3_brake
3_brake_eeg.fif already exists, skipping
reading 3_turn
3_turn_eeg.fif already exists, skipping
reading 4_change
4_change_eeg.fif already exists, skipping
reading 4_throttle
4_throttle_eeg.fif already exists, skipping
reading 4_brake
4_brake_eeg.fif already exists, skipping
reading 4_turn
4_turn_eeg.fif already exists, skipping
reading 5_change
5_change_eeg.fif already

In [18]:
# event = {139, 141, 145, 125, 127, 129, 131, 138, 143}

# event_ids = {}

# for i in range(1, 2):
#     for j in labels:
#         root = '/run/media/aaditya/DATA/Datasets/'
#         file = f'{i}_{j}'
#         proc_filepath = root + 'intermediate/pre_epoch/' + file + '_eeg.fif'

#         print(f'\nreading {proc_filepath}')
#         proc_file = mne.io.read_raw_fif(proc_filepath)

#         events, events_dict = mne.events_from_annotations(proc_file)
#         for k, v in events_dict.items():
#             try:
#                 x = int(k)
#                 if x in event:
#                     event_ids[x] = v
#             except:
#                 pass
        
#         proc_file.close()
#         del proc_file


In [23]:
print('\ncreating epochs\n')

event_mapping = {
    'front emergency brake': 139,
    'side front cut-in': 141,
    'pedestrian crossing': 145,
    'Left-turn sign': 125,
    'Right-turn sign': 127,
    'Static obstacle on the right': 129,
    'Static obstacle on the left': 131,
    'overtaking': 137,
    'Congestion relief': 143
}

#need to apply category offsets so that all events have unique ids
category_offsets = {
    'brake': 0,
    'turn': 1000,
    'change': 2000,
    'throttle': 3000
}


event_mapping = {v:k for k,v in event_mapping.items()}

for i in range(1, 31):
    eps = []
    out_filepath = root + 'intermediate/post_epoch/' + f'{i}' + '_epo.fif'
    if os.path.isfile(out_filepath):
        print(f'{i}_epo.fif already exists, skipping')
        continue

    for j in labels:
        root = '/run/media/aaditya/DATA/Datasets/'
        file = f'{i}_{j}'

        print(f'reading {file}_eeg.fif')
        proc_filepath = root + 'intermediate/pre_epoch/' + file + '_eeg.fif'
        proc_file = mne.io.read_raw_fif(proc_filepath)

        #need to do this cuz otherwise the mne generated event ids clash across different set files
        #and anyways this is how the original does it
        if j == 'brake':
            rel_events = {139, 141, 145}
        elif j == 'turn':
            rel_events = {125, 127}
        elif j == 'change':
            rel_events = {129, 131}
        elif j == 'throttle':
            rel_events = {137, 143}

        #get event dict
        events, event_dict = mne.events_from_annotations(proc_file)

        event_ids = {}
        for k, v in event_dict.items():
            try:
                x = int(k)
            except:
                continue
            if x in rel_events:
                new_id = v + category_offsets[j]
                event_ids[event_mapping[x]] = new_id
                events[events[:, 2] == v, 2] = new_id    

        print(event_ids)
        if not event_ids:
            continue
        
        #check preprocessing.m for event code definitions
        #tmin to tmax defines the epoch length in time
        #baseline defines the baseline voltage from 500ms before the event upto the event
        epoch = mne.Epochs(proc_file, events, event_id=event_ids,
                            tmin=-0.5, tmax=1.5, baseline=(-0.5, None))
        eps.append(epoch)
    
    final_epoch = mne.concatenate_epochs(eps)
    final_epoch.save(out_filepath)
    print(f'epochs written for sample {i}: {i}_epo.fif')

    del eps


creating epochs

1_epo.fif already exists, skipping
2_epo.fif already exists, skipping
3_epo.fif already exists, skipping
4_epo.fif already exists, skipping
5_epo.fif already exists, skipping
6_epo.fif already exists, skipping
7_epo.fif already exists, skipping
8_epo.fif already exists, skipping
9_epo.fif already exists, skipping
10_epo.fif already exists, skipping
11_epo.fif already exists, skipping
12_epo.fif already exists, skipping
13_epo.fif already exists, skipping
14_epo.fif already exists, skipping
15_epo.fif already exists, skipping
16_epo.fif already exists, skipping
17_epo.fif already exists, skipping
18_epo.fif already exists, skipping
19_epo.fif already exists, skipping
20_epo.fif already exists, skipping
21_epo.fif already exists, skipping
22_epo.fif already exists, skipping
23_epo.fif already exists, skipping
24_epo.fif already exists, skipping
25_epo.fif already exists, skipping
26_epo.fif already exists, skipping
reading 27_change_eeg.fif
{'Static obstacle on the righ

In [24]:
print('\napplying ICA\n')


for i in range(1, 31):
    root = '/run/media/aaditya/DATA/Datasets/'
    ica_out_dir = root + 'intermediate/post_ica/'
    post_epoch_filepath = root + 'intermediate/post_epoch/' + f'{i}_epo.fif'
    ica_out_filepath = ica_out_dir + f'{i}_ica.fif'

    if os.path.isfile(ica_out_filepath):
        print(f'{i}_ica.fif already exists, skipping')
        continue

    print(f'Processing {i}_epo.fif')
    epochs = mne.read_epochs(post_epoch_filepath, preload=True)

    ica = mne.preprocessing.ICA(n_components=20, random_state=42, max_iter=800)
    ica.fit(epochs)

    epochs_ica = ica.apply(epochs)

    epochs_ica.save(ica_out_filepath, overwrite=True)
    print(f'ICA applied and epochs saved for sample {i}: {i}_ica.fif')

    del epochs, ica, epochs_ica



applying ICA

1_ica.fif already exists, skipping
2_ica.fif already exists, skipping
3_ica.fif already exists, skipping
4_ica.fif already exists, skipping
5_ica.fif already exists, skipping
6_ica.fif already exists, skipping
7_ica.fif already exists, skipping
8_ica.fif already exists, skipping
9_ica.fif already exists, skipping
10_ica.fif already exists, skipping
11_ica.fif already exists, skipping
12_ica.fif already exists, skipping
13_ica.fif already exists, skipping
14_ica.fif already exists, skipping
15_ica.fif already exists, skipping
16_ica.fif already exists, skipping
17_ica.fif already exists, skipping
18_ica.fif already exists, skipping
19_ica.fif already exists, skipping
20_ica.fif already exists, skipping
21_ica.fif already exists, skipping
22_ica.fif already exists, skipping
23_ica.fif already exists, skipping
24_ica.fif already exists, skipping
25_ica.fif already exists, skipping
26_ica.fif already exists, skipping
Processing 27_epo.fif
ICA applied and epochs saved for sam